In [ ]:
# 1. Clean install
!rm -rf torchinspector
!git clone https://github.com/blackcat312340/torchinspector.git
%cd /content/torchinspector
!pip install -e .
%cd /content

from torchinspector import Inspector, __version__
import torch
print(f"TorchInspector v{__version__}")
print(f"PyTorch {torch.__version__}")

In [ ]:
# 2. Setup TensorBoard
%load_ext tensorboard
!rm -rf runs/mnist_demo

In [ ]:
# 3. Train CNN on MNIST
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

model = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(32 * 7 * 7, 10),
)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

transform = transforms.Compose([transforms.ToTensor()])
train_ds = datasets.MNIST('data', train=True, download=True, transform=transform)
loader = DataLoader(train_ds, batch_size=64, shuffle=True)

ins = Inspector(model, opt, 'runs/mnist_demo',
                log_interval=50,
                feature_map_interval=200,
                feature_map_channels=8,
                weight_heatmap_interval=200,
                health_report_interval=200)

watched = ins.watch_auto(max_layers=6)
print(f"Watching: {watched}")

step = 0
for epoch in range(2):
    for x, y in loader:
        opt.zero_grad()
        loss = nn.functional.cross_entropy(model(x), y)
        loss.backward()
        opt.step()
        step += 1
        ins.step(loss=loss.item())
        if step % 500 == 0 and step > 0:
            ins.explain(x[:1], method='gradcam', target=y[0].item())

ins.close()
print(f'Done! {step} steps')

In [ ]:
%tensorboard --logdir=runs/mnist_demo